# DSCI 552 - Homework 3

**Name:** Tina Arezoomanians  
**GitHub Username:** tinaarezoomanian

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import bootstrap

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) AReM Data Set

The AReM data set contains 7 activity folders and 88 instances. Each instance contains 6 time series with 480 consecutive values.

In [2]:
data_path = "data/AReM"

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

for activity in activities:
    folder = os.path.join(data_path, activity)
    files = [f for f in os.listdir(folder) if f.endswith(".csv")]
    print(activity, len(files))

bending1 7
bending2 6
cycling 15
lying 15
sitting 15
standing 15
walking 15


### (b) Train and Test Data

For bending1 and bending2, datasets 1 and 2 are used as test data. For the other activities, datasets 1, 2, and 3 are used as test data. The remaining datasets are used as training data.

In [3]:
train_files = []
test_files = []

for activity in activities:
    folder = os.path.join(data_path, activity)
    
    files = sorted(
        [f for f in os.listdir(folder) if f.endswith(".csv")],
        key=lambda x: int(x.replace("dataset", "").replace(".csv", ""))
    )
    
    for file in files:
        dataset_num = int(file.replace("dataset", "").replace(".csv", ""))
        file_path = os.path.join(folder, file)
        
        if activity in ["bending1", "bending2"]:
            if dataset_num in [1, 2]:
                test_files.append((activity, file_path))
            else:
                train_files.append((activity, file_path))
        else:
            if dataset_num in [1, 2, 3]:
                test_files.append((activity, file_path))
            else:
                train_files.append((activity, file_path))

print("Training instances:", len(train_files))
print("Test instances:", len(test_files))
print("Total instances:", len(train_files) + len(test_files))

Training instances: 69
Test instances: 19
Total instances: 88


### (c) Feature Extraction

#### (i) Time-Domain Features

Some common time-domain features used for time series classification are:

- Minimum
- Maximum
- Mean
- Median
- Standard deviation
- Variance
- First quartile
- Third quartile
- Range
- Interquartile range

In [4]:
sample_file = train_files[0][1]

with open(sample_file, "r") as f:
    for _ in range(10):
        print(f.readline().strip())

# Task: bending1
# Frequency (Hz): 20
# Clock (millisecond): 250
# Duration (seconds): 120
# Columns: time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,42.00,0.71,21.25,0.43,30.00,0.00
250,41.50,0.50,20.25,1.48,31.25,1.09
500,41.50,0.50,14.25,1.92,33.00,0.00
750,40.75,0.83,15.75,0.43,33.00,0.00
1000,40.00,0.71,20.00,2.74,32.75,0.43


In [5]:
column_names = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

time_series_cols = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

sample_df = pd.read_csv(
    sample_file,
    skiprows=5,
    header=None,
    names=column_names
)

print(sample_df.shape)
sample_df.head()

(480, 7)


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0,42.00,0.71,21.25,0.43,30.00,0.00
1,250,41.50,0.50,20.25,1.48,31.25,1.09
2,500,41.50,0.50,14.25,1.92,33.00,0.00
3,750,40.75,0.83,15.75,0.43,33.00,0.00
4,1000,40.00,0.71,20.00,2.74,32.75,0.43


#### (ii) Extract Time-Domain Features

In [6]:
from io import StringIO

def extract_features(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()

    # Remove accidental trailing commas
    lines = [line.rstrip().rstrip(",") + "\n" for line in lines]

    data_text = "".join(lines)

    # dataset4 in bending2 is space-separated
    if "bending2/dataset4.csv" in file_path:
        df = pd.read_csv(
            StringIO(data_text),
            skiprows=5,
            header=None,
            names=column_names,
            sep=r"\s+"
        )
    else:
        df = pd.read_csv(
            StringIO(data_text),
            skiprows=5,
            header=None,
            names=column_names
        )

    features = {}

    for i, col in enumerate(time_series_cols, start=1):
        features[f"min{i}"] = df[col].min()
        features[f"max{i}"] = df[col].max()
        features[f"mean{i}"] = df[col].mean()
        features[f"median{i}"] = df[col].median()
        features[f"std{i}"] = df[col].std()
        features[f"1st_quart{i}"] = df[col].quantile(0.25)
        features[f"3rd_quart{i}"] = df[col].quantile(0.75)

    return features

In [7]:
all_files = train_files + test_files

feature_rows = []

for activity, file_path in all_files:
    features = extract_features(file_path)
    features["activity"] = activity
    features["file"] = os.path.basename(file_path)
    feature_rows.append(features)

feature_df = pd.DataFrame(feature_rows)

print("Number of instances:", feature_df.shape[0])
print("Number of extracted features:", feature_df.shape[1] - 2)

feature_df.head()

Number of instances: 88
Number of extracted features: 42


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6,activity,file
0,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,36.50,0.0,1.79,0.493292,0.43,0.513506,0.00,0.94,bending1,dataset3.csv
1,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,36.33,0.0,2.18,0.613521,0.50,0.524317,0.00,1.00,bending1,dataset4.csv
2,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,31.25,0.0,1.79,0.383292,0.43,0.389164,0.00,0.50,bending1,dataset5.csv
3,37.00,48.00,43.454958,43.25,1.386098,42.50,45.00,0.0,1.58,0.378083,...,24.00,0.0,5.26,0.679646,0.50,0.622534,0.43,0.87,bending1,dataset6.csv
4,36.25,48.00,43.969125,44.50,1.618364,43.31,44.67,0.0,1.50,0.413125,...,23.75,0.0,2.96,0.555312,0.49,0.487826,0.00,0.83,bending1,dataset7.csv


#### (iii) Standard Deviations and Bootstrap Confidence Intervals

In [8]:
feature_columns = feature_df.columns[:42]

results = []

for feature in feature_columns:
    values = feature_df[feature].to_numpy()
    
    std_estimate = np.std(values, ddof=1)
    
    boot_result = bootstrap(
        (values,),
        lambda x: np.std(x, ddof=1),
        confidence_level=0.90,
        n_resamples=10000,
        method="percentile",
        random_state=42
    )
    
    results.append({
        "Feature": feature,
        "Standard Deviation": std_estimate,
        "90% CI Lower": boot_result.confidence_interval.low,
        "90% CI Upper": boot_result.confidence_interval.high
    })

bootstrap_results = pd.DataFrame(results)

bootstrap_results

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.569975,8.258314,10.736581
1,max1,4.394362,3.333388,5.300986
2,mean1,5.335718,4.712411,5.877456
3,median1,5.440054,4.799744,5.995530
4,std1,1.772153,1.568692,1.951558
5,1st_quart1,6.153590,5.568990,6.640666
6,3rd_quart1,5.138925,4.321356,5.848241
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.619850,5.397569
9,mean2,1.574164,1.396172,1.704331


#### (iv) Most Important Time-Domain Features

I think the three most important time-domain features are mean, standard deviation, and maximum. The mean shows the overall level of the time series, the standard deviation shows how much the values change, and the maximum shows the highest value reached. Together, these features give information about the general level, variation, and range of the activity.

## 2. ISLR 3.7.4

### (a)

The cubic regression should have a training RSS that is less than or equal to the linear regression. Even though the true relationship is linear, the cubic model is more flexible and can fit the training data at least as well as the linear model.

### (b)

For the test RSS, I would expect the linear regression to have a lower RSS because the true relationship is linear. The cubic model may fit some of the noise in the training data, which can make it perform worse on new data.

### (c)

The cubic regression should have a training RSS that is less than or equal to the linear regression. Since the cubic model is more flexible, it can fit the training data better, especially when the true relationship is not linear.

### (d)

There is not enough information to tell which model will have a lower test RSS. It depends on how nonlinear the true relationship is. If it is close to linear, the linear model could perform better. If the relationship is closer to a cubic shape, the cubic model could perform better.